# Future-Guided Learning — CHB-MIT (Kaggle Edition)

Tái hiện thực nghiệm EEG của bài báo:
> Gunasekaran et al., *A predictive approach to enhance time-series forecasting*, Nature Communications 2025.
> Repo gốc: https://github.com/SkyeGunasekaran/FutureGuidedLearning

**Notebook này clone từ fork cá nhân** (`github.com/kkyaha/Forecast`) — đã có sẵn 1 cải
tiến tự đề xuất: **Bidirectional LSTM** cho `CNN_LSTM_Model` (Teacher + Student/FGL), bật
qua cờ `--bidirectional` ở Stage 1 và Stage 3 bên dưới. Chi tiết patch:
`repro/ours/patches/chbmit_bidirectional_lstm.patch` (trong repo). Stage 2 (Baseline)
giữ nguyên code gốc tác giả, không đổi.

---
### ⚙️ Cấu hình Kaggle Notebook:
1. **Accelerator:** Chọn **GPU T4 x2** hoặc **GPU P100**.
2. **Internet:** Bật **Internet: ON**.


## 1. Clone repository về Kaggle Working Directory

In [ ]:
import os
import shutil

PROJECT_DIR = '/kaggle/working/Forecast'
REPO_URL = 'https://github.com/kkyaha/Forecast.git'

if not os.path.isdir(PROJECT_DIR):
    print('Dang clone repo tu GitHub...')
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print('Repo da co trong /kaggle/working.')

%cd {PROJECT_DIR}/repro/build/CHBMIT
!pwd


## 2. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q mne==1.5.1 hickle stft tqdm scikit-learn

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

## 3. Nạp dữ liệu CHB-MIT (Chọn 1 trong 2 cách)

### 👉 CÁCH 1 (Khuyên dùng - CỰC NHANH): Dùng Dataset có sẵn trên Kaggle
1. Bấm nút **"+ Add Input"** ở góc trên bên phải notebook.
2. Tìm từ khóa **`chb-mit`** hoặc **`chb01`** và bấm dấu **`+`** để thêm dataset vào notebook.
3. Chạy cell bên dưới: code sẽ tự động quét và tạo đường dẫn tắt (symlink) trong 1 giây, **không cần tải về**!

In [ ]:
import os
import glob

# Kiểm tra các dataset đã thêm vào /kaggle/input
print('Các dataset có trong /kaggle/input:')
!ls -l /kaggle/input

# Tự động tìm thư mục chb01 hoặc các file edf trong /kaggle/input
found_chb01 = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'chb01' in dirs:
        found_chb01 = os.path.join(root, 'chb01')
        break
    # Hoặc nếu các file chb01_*.edf nằm trực tiếp trong một thư mục
    if any(f.startswith('chb01_') and f.endswith('.edf') for f in files):
        found_chb01 = root
        break

if found_chb01:
    print(f'\n[TÌM THẤY] Dữ liệu chb01 tại: {found_chb01}')
    dest_dir = 'Dataset/chb01'
    os.makedirs('Dataset', exist_ok=True)
    if os.path.islink(dest_dir) or os.path.exists(dest_dir):
        !rm -rf {dest_dir}
    os.symlink(found_chb01, dest_dir)
    print(f'-> Đã liên kết (symlink) {found_chb01} vào {dest_dir} thành công!')
    !ls -lh Dataset/chb01 | head -5
else:
    print('\n[CHƯA THẤY] Bạn chưa bấm "+ Add Input" bộ CHB-MIT nào.')
    print('Nếu không muốn thêm qua Kaggle, bạn có thể chạy Cách 2 bên dưới để tải từ PhysioNet.')

### 👉 CÁCH 2 (Dự phòng): Tải trực tiếp từ PhysioNet qua wget
*(Chỉ chạy cell này nếu bạn KHÔNG dùng Cách 1 ở trên)*

In [ ]:
# Chỉ chạy khi chưa có Dataset/chb01
if not os.path.exists('Dataset/chb01') or not any(f.endswith('.edf') for f in os.listdir('Dataset/chb01')):
    print('Đang tải dữ liệu chb01 từ PhysioNet (có thể mất vài phút)...')
    !wget -r -N -c -np -nH --cut-dirs=3 -A edf -P Dataset "https://physionet.org/files/chbmit/1.0.0/chb01/"
    !ls -lh Dataset/chb01 | head -5
else:
    print('chb01 đã sẵn sàng, không cần tải lại!')

## 4. Stage 1 — Train Teacher (Seizure Detection)

Mô hình phát hiện cơn động kinh (ictal vs interictal). Trọng số lưu tại `pytorch_models/Patient_1_detection`.

In [ ]:
!python -m exp.seizure_detection --patient 1 --epochs 50 --patience 5


### ⭐ Biến thể Bidirectional LSTM (dành cho báo cáo CUỐI KỲ, không dùng ở báo cáo giữa kỳ)

Cell dưới đây chạy lại Stage 1 với cải tiến tự đề xuất LSTM 2 chiều
(`repro/ours/patches/chbmit_bidirectional_lstm.patch`). **Để mặc định không chạy** (dòng lệnh
bị comment) — bỏ dấu `#` ở đầu dòng nếu bạn đang làm báo cáo cuối kỳ và muốn lấy số liệu này.

In [ ]:
# !python -m exp.seizure_detection --patient 1 --epochs 50 --patience 5 --bidirectional


## 5. Stage 2 — Train Baseline (Seizure Prediction, không có FGL)

In [ ]:
!python -m exp.seizure_prediction --patient 1 --model CNN_LSTM --trials 3 --epochs 25

## 6. Stage 3 — Train Student + FGL (Seizure Prediction, CÓ Teacher dẫn đường)

In [ ]:
!python -m exp.FGL_CHBMIT --patient 1 --epochs 30 --trials 3 --optimizer Adam --alpha 0.5 --temperature 4


### ⭐ Biến thể Bidirectional LSTM (dành cho báo cáo CUỐI KỲ, không dùng ở báo cáo giữa kỳ)

Tương tự Stage 1 — bỏ dấu `#` ở dòng lệnh dưới đây nếu muốn lấy số liệu cải tiến cho báo cáo
cuối kỳ.

In [ ]:
# !python -m exp.FGL_CHBMIT --patient 1 --epochs 30 --trials 3 --optimizer Adam --alpha 0.5 --temperature 4 --bidirectional


## 7. Tổng kết kết quả & Xuất Checkpoints để tải về máy

In [ ]:
import os
import shutil

print('\n=================== KẾT QUẢ ĐỐI CHIẾU ===================')
if os.path.exists('Detection_results.txt'):
    with open('Detection_results.txt') as f:
        print('--- Teacher (Detection) Results ---')
        print(f.read().strip())

if os.path.exists('Prediction_results.txt'):
    with open('Prediction_results.txt') as f:
        print('\n--- Baseline (Prediction without FGL) Results ---')
        print(f.read().strip())

print('\n(Kết quả Student FGL được in trực tiếp ở cell Stage 3 ở trên)')
print('=========================================================')

# Nén thư mục models và các file kết quả thành file zip trong /kaggle/working
zip_output_path = '/kaggle/working/fgl_chbmit_patient01_artifacts'
if os.path.exists('pytorch_models'):
    shutil.make_archive(zip_output_path, 'zip', root_dir='.', base_dir='pytorch_models')
    print(f'\n[THÀNH CÔNG] Đã nén weights mô hình vào: {zip_output_path}.zip')
    print('👉 Bạn hãy bấm vào panel bên phải (Data > Output) để tải file zip về máy!')